# 26_02 IsolationForest 이상탐지 개념코드

In [12]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

✅ 환경 설정 완료! 현재 적용된 폰트: ['Malgun Gothic']


새로운 패키지 설치가 필요합니다.

```bash
pip install scikit-learn
```

In [13]:
from sklearn.ensemble import IsolationForest

In [14]:
df = pd.read_csv("26_mimii_features.csv")

In [15]:
# 만들기 → fit → predict 3단계
X = df[["rms", "spectral_centroid", "zero_crossing_rate"]]

# 1. 만들기
iso = IsolationForest(contamination=0.18, random_state=42)  

# 학습 (보통 한 번만)
iso.fit(X)

# 예측
pred = iso.predict(X)
print(set(pred))

# 결과
# {np.int64(1), np.int64(-1)}
# 해석
# 1 : 정상 데이터, inlier
# -1 : 이상치, outlier
# 정상 또는 이상치 중에 하나일거라 말해줌

{np.int64(1), np.int64(-1)}


In [16]:
X = df[["rms", "spectral_centroid", "zero_crossing_rate"]]
iso = IsolationForest(contamination=0.18, random_state=42)
iso.fit(X)
pred = iso.predict(X)
print(set(pred)) # -1과 1 두 값만 존재 (-1=이상, 1=정상)

# -1(이상)을 1로, 1(정상)을 0으로
pred_anom = (pred == -1).astype(int)
print(int(pred_anom.sum()))

{np.int64(1), np.int64(-1)}
40


In [17]:
# 정답과 교차표 (채점)
print(pd.crosstab(df["label"], pred_anom).to_dict())

{0: {0: 166, 1: 14}, 1: {0: 14, 1: 26}}


In [18]:
# 점수가 작은 순서가 곧 이상 우선순위
score = iso.score_samples(X)
print(round(float(score.min()), 3), round(float(score.max()), 3))
# -0.667 -0.372   (작을수록 이상)

-0.678 -0.372


In [19]:
# contamination을 바꾸면
# 비율을 키울수록 더 많이 잡힘
for c in [0.05, 0.10, 0.18, 0.25]:
    p = IsolationForest(contamination=c, random_state=42).fit_predict(X)
    n = int((p == -1).sum())
    real = int(((p == -1) & (df["label"] == 1)).sum())
    print(c, n, real)
# 출력: 0.05 11 10 / 0.10 22 16 / 0.18 40 27 / 0.25 55 35

0.05 11 10
0.1 22 14
0.18 40 26
0.25 55 35


In [20]:
# StandardScaler 전후 (트리 기반은 영향 작음)
# Xs = StandardScaler().fit_transform(X)
# ps = IsolationForest(contamination=0.18, random_state=42).fit_predict(Xs)
# real_s = int(((ps == -1) & (df["label"] == 1)).sum())
# print(real_s)
# 출력: 27   (스케일링 전 27과 거의 같음)

In [21]:
# 학습된 모델을 새 데이터에 적용 (예측만)
new = pd.read_csv("26_mimii_new.csv")
X_new = new[["rms", "spectral_centroid", "zero_crossing_rate"]]  # 같은 특징·순서
pred_new = iso.predict(X_new)        # 다시 학습하지 않고 predict만
print(int((pred_new == -1).sum()))
# 출력: 10   (label 없는 새 데이터에서 약 10개 이상)

8


# 실습

In [22]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [23]:
df = pd.read_csv("26_mimii_features.csv")
feats = ["rms", "spectral_centroid", "zero_crossing_rate"]

### 실습 1 — IsolationForest 첫 적용

In [24]:
X = df[feats]                                  # 특징만 (label 제외)
iso = IsolationForest(contamination=0.18, random_state=42)
iso.fit(X)
pred = iso.predict(X)
print(set(pred))

{np.int64(1), np.int64(-1)}


### 실습 2 — 이상 개수 세어보기 + 교차표

In [25]:
df["pred_anom"] = (pred == -1).astype(int)     # -1(이상)→1, 1(정상)→0
print(int(df["pred_anom"].sum()))              # 40
print(pd.crosstab(df["label"], df["pred_anom"]))   # 실제 이상 40 중 27 맞힘

40
pred_anom    0   1
label             
0          166  14
1           14  26


### 실습 3 — 이상 점수 정렬해 Top N

In [26]:
df["score"] = iso.score_samples(X)             # 작을수록 이상
top = df.sort_values("score").head(10)         # 가장 이상한 10개
print(top[feats + ["score", "label"]])
print(int(top["label"].sum()))                 # 상위 10개 중 실제 이상 수

        rms  spectral_centroid  zero_crossing_rate     score  label
70   0.1599             1947.0              0.0891 -0.678462      1
54   0.0144             1864.1              0.1384 -0.638786      0
108  0.1467             2674.4              0.1037 -0.625262      1
157  0.1273             1577.0              0.0811 -0.613582      1
149  0.1303             2126.7              0.1419 -0.610869      1
81   0.0757             3055.9              0.1451 -0.608784      1
159  0.0670             2703.1              0.1722 -0.600776      1
23   0.1250             2846.4              0.0687 -0.598813      1
90   0.1259             2910.5              0.1078 -0.597624      1
218  0.1247             2783.3              0.0794 -0.583770      1
9


### 실습 4 — contamination 값 실험

In [27]:
rows = []
for c in [0.05, 0.10, 0.18, 0.25]:
    p = IsolationForest(contamination=c, random_state=42).fit_predict(X)
    n = int((p == -1).sum())
    real = int(((p == -1) & (df["label"] == 1)).sum())
    rows.append({"contamination": c, "잡힌개수": n, "진짜이상": real})
print(pd.DataFrame(rows))                       # 11/22/40/55, 진짜 10/16/27/35

   contamination  잡힌개수  진짜이상
0           0.05    11    10
1           0.10    22    14
2           0.18    40    26
3           0.25    55    35


### 실습 5 — StandardScaler 적용 전후 비교

In [28]:
real_raw = int(((iso.predict(X) == -1) & (df["label"] == 1)).sum())   # 약 27
Xs = StandardScaler().fit_transform(X)
ps = IsolationForest(contamination=0.18, random_state=42).fit_predict(Xs)
real_scaled = int(((ps == -1) & (df["label"] == 1)).sum())            # 약 27
print(pd.DataFrame([{"구분": "원본", "진짜이상": real_raw},
                    {"구분": "스케일링", "진짜이상": real_scaled}]))

     구분  진짜이상
0    원본    26
1  스케일링    26


### 실습 6 — 새 데이터에 모델 적용

In [29]:
new = pd.read_csv("26_mimii_new.csv")
X_new = new[feats]                              # 학습 때와 같은 특징·순서
pred_new = iso.predict(X_new)                   # 다시 학습하지 않고 예측만
print(int((pred_new == -1).sum()))              # 약 10
new["score"] = iso.score_samples(X_new)
print(new.sort_values("score").head(5)[feats + ["score"]])

8
       rms  spectral_centroid  zero_crossing_rate     score
13  0.1285             2974.6              0.1216 -0.633566
31  0.0629             3028.4              0.1203 -0.573328
6   0.1229             2668.5              0.0964 -0.557898
20  0.1286             1901.8              0.0926 -0.543498
0   0.1104             2565.5              0.1073 -0.526070


### 실습 7 — 두 방법 결과 교차 확인

In [30]:
normal = df[df["label"] == 0]
z_any = pd.Series(False, index=df.index)
for f in feats:                                 # 모듈 1 방식: Z-score |z|>3 OR
    z = (df[f] - normal[f].mean()) / normal[f].std()
    z_any = z_any | (np.abs(z) > 3)
df["z_anom"] = z_any.astype(int)
df["if_anom"] = df["pred_anom"]
print(pd.crosstab(df["z_anom"], df["if_anom"]))
# IsolationForest만 잡은(=z_anom 0, if_anom 1) 데이터 = 관계형(다변량) 이상 후보
only_if = df[(df["z_anom"] == 0) & (df["if_anom"] == 1)]
print(len(only_if))

if_anom    0   1
z_anom          
0        168  13
1         12  27
13
